# How to run one action across vendors with Operations

Getting the running configuration means a different task on each platform: one for Cisco IOS, one for Arista EOS, one for Juniper Junos. Without Operations, the calling code selects the task with `if host.platform == ...`, and learns the shape of `Result.result` by reading each task.

An Operation defines the action once, with a typed input and a typed output. Packages implement it with plain tasks, one per platform and connection. You run the Operation once over the whole inventory: each host runs the task registered for its platform, and its result holds an instance of the output model, which your type checker knows.

The tasks below return fixed text instead of talking to devices, so the guide runs anywhere.

## Set up

The inventory has one host per platform: `ios`, `eos`, `junos` and `panos`. The configuration selects `SerialRunner` and sets `core.operation_connection`, the connection used when a call does not give one. Paths are relative to the notebook working directory, `docs/howto/`.

In [1]:
from dataclasses import dataclass

from nornir import InitNornir
from nornir.core.operations import Operation, OperationRegistry
from nornir.core.task import Result, Task

nr = InitNornir(config_file="operations/config.yaml")
for name, host in nr.inventory.hosts.items():
    print(f"{name}: {host.platform}")
print(f"Default connection: {nr.config.core.operation_connection}")

core-1: ios
core-2: eos
edge-1: junos
fw-1: panos
Default connection: napalm


## Define the Operation

An Operation is a class with a `name` and two type arguments: the input model and the output model. Any class that can be built from keyword arguments works as a model; this guide uses dataclasses. A Pydantic model also works, and Nornir then validates the input with its `model_validate` method.

The Operation holds no vendor logic. It is usually published in a package that several implementation packages depend on.

In [2]:
@dataclass
class GetConfigInput:
    retrieve: str = "running"
    sanitized: bool = False


@dataclass
class GetConfigOutput:
    config: str


class GetConfig(Operation[GetConfigInput, GetConfigOutput]):
    name = "get_config"

## Implement it for each platform

An implementation is a plain task: it receives the task and the validated input as `params`, and returns an instance of the output model. You can run it on its own with `nr.run(task=..., params=...)`.

Each task is registered for one `(operation, platform, connection)` key. A package declares its tasks as entry points, so installing the package is enough:

```toml
[project.entry-points."nornir.plugins.operation_tasks"]
"get_config.ios.napalm" = "my_package.operations:ios_get_config"
```

In a script or a notebook, register them with `OperationRegistry.register`:

In [3]:
def ios_get_config(task: Task, params: GetConfigInput) -> GetConfigOutput:
    return GetConfigOutput(config=f"hostname {task.host.name}\n! {params.retrieve} (IOS)")


def eos_get_config(task: Task, params: GetConfigInput) -> GetConfigOutput:
    return GetConfigOutput(config=f"hostname {task.host.name}\n! {params.retrieve} (EOS)")


def junos_get_config(task: Task, params: GetConfigInput) -> GetConfigOutput:
    return GetConfigOutput(config=f"host-name {task.host.name}; /* {params.retrieve} (Junos) */")


for platform, task in [("ios", ios_get_config), ("eos", eos_get_config), ("junos", junos_get_config)]:
    OperationRegistry.register(operation="get_config", platform=platform, connection="napalm", task=task)

## Run it across the inventory

`nr.run_operation()` returns a stream, as `nr.run_stream()` does, and `results()` yields the final result of each host. The loop has no condition on the platform. Because the Operation class is passed, the type checker types `res.result` as `GetConfigOutput`, so `res.result.config` is a `str`.

Nothing is registered for `panos`, so `fw-1` fails with `OperationNotRegisteredError`, which lists the platforms that are registered. The other hosts are not affected.

In [4]:
for res in nr.run_operation(GetConfig, GetConfigInput(retrieve="startup")).results():
    if res.failed:
        print(f"{res.host}: FAILED: {res.exception}")
    else:
        print(f"{res.host}:\n{res.result.config}")

core-1:
hostname core-1
! startup (IOS)
core-2:
hostname core-2
! startup (EOS)
edge-1:
host-name edge-1; /* startup (Junos) */
fw-1: FAILED: no task registered for operation 'get_config' on platform 'panos' with connection 'napalm' (host 'fw-1'). Registered for 'get_config': eos.napalm, ios.napalm, junos.napalm.


The input is validated before any host starts. A field name with a typo raises `OperationInputError` from `run_operation` itself:

In [5]:
from nornir.core.exceptions import OperationInputError

try:
    nr.run_operation(GetConfig, {"sanitize": True}, connection="napalm")
except OperationInputError as error:
    print(error)

invalid input for operation 'get_config': unexpected field 'sanitize' (did you mean 'sanitized'?)


## Give each host its own input

Pass a function that receives the host and returns its input. Nornir calls it when the host starts. If it raises for one host, only that host fails.

In [6]:
retrieve = {"core-1": "startup"}
stream = nr.filter(filter_func=lambda host: host.name != "fw-1").run_operation(
    GetConfig, lambda host: GetConfigInput(retrieve=retrieve.get(host.name, "running"))
)
for res in stream.results():
    print(f"{res.host}: {res.result.config.splitlines()[-1]}")

core-1: ! startup (IOS)
core-2: ! running (EOS)
edge-1: host-name edge-1; /* running (Junos) */


## Report a change

A task that changes the device returns a `Result` that holds the output, with `changed` and `diff` set. Nornir keeps both, so `print_result` and other tools that read them keep working.

In [7]:
@dataclass
class SetBannerInput:
    text: str


@dataclass
class SetBannerOutput:
    previous: str


class SetBanner(Operation[SetBannerInput, SetBannerOutput]):
    name = "set_banner"


def eos_set_banner(task: Task, params: SetBannerInput) -> Result[SetBannerOutput]:
    previous = "Welcome"
    return Result(
        host=task.host,
        result=SetBannerOutput(previous=previous),
        changed=params.text != previous,
        diff=f"-banner {previous}\n+banner {params.text}",
    )


OperationRegistry.register(operation="set_banner", platform="eos", connection="napalm", task=eos_set_banner)

for res in nr.filter(platform="eos").run_operation(SetBanner, SetBannerInput(text="Authorized access only")).results():
    print(f"{res.host}: changed={res.changed}, previous={res.result.previous!r}")
    print(res.diff)

core-2: changed=True, previous='Welcome'
-banner Welcome
+banner Authorized access only


## List the known Operations

`OperationRegistry.operations()` lists every Operation, its models and its implementations, with the kind of each task and where it was registered from. It calls no task, so a tool can use it to show what is installed.

In [8]:
for info in OperationRegistry.operations():
    print(f"{info.name}: {info.input_model.__name__} -> {info.output_model.__name__}")
    for registration in info.implementations:
        print(f"    {registration.platform}.{registration.connection}: {registration.kind}, {registration.source}")

get_config: GetConfigInput -> GetConfigOutput
    eos.napalm: sync, __main__.eos_get_config
    ios.napalm: sync, __main__.ios_get_config
    junos.napalm: sync, __main__.junos_get_config
set_banner: SetBannerInput -> SetBannerOutput
    eos.napalm: sync, __main__.eos_set_banner
